In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from ydata_profiling import ProfileReport
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
df = sns.load_dataset('titanic')

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
profile = ProfileReport()
profile = ProfileReport(df, title="Titanic Dataset Profiling Report", explorative=True)

In [ ]:
profile

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='sex', hue='survived', data=df)

plt.title('Survival by Gender')

plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='pclass', hue='survived', data=df)

plt.title('Survival by Passenger Class')

plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['age'], bins=30, kde=True)

plt.title('Age Distribution')

plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='survived', y='age', data=df)

plt.title('Age vs Survival')

plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['fare'], bins=30, kde=True)

plt.title('Fare Distribution')

plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=['number'])

plt.figure(figsize=(8,6))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')

plt.title('Correlation Matrix')

plt.show()

In [ ]:
df.drop(columns=[
    'alive',
    'deck',
    'embark_town',
    'adult_male',
    'class'
], inplace=True)


In [ ]:
df['age'].fillna(df['age'].median(), inplace=True)

In [ ]:
df['embarked'].value_counts()

In [ ]:
df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
le = LabelEncoder()
df['sex'] = le.fit_transform(df['sex'])

In [ ]:
df = pd.get_dummies(df, columns=['embarked', 'who'], drop_first=True)

In [ ]:
df['embarked_Q'] = df['embarked_Q'].astype(int)
df['embarked_S'] = df['embarked_S'].astype(int)
df['who_man'] = df['who_man'].astype(int)
df['who_woman'] = df['who_woman'].astype(int)
df['alone'] = df['alone'].astype(int)


In [ ]:
df['fare_log'] = np.log1p(df['fare'])

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['fare_log'], bins=30, kde=True)
plt.title("Fare Distribution After Log Transformation")
plt.show()

In [ ]:
df.drop(columns=['fare'], inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.info()

In [ ]:
profile = ProfileReport()
profile = ProfileReport(df, title="Titanic Dataset Profiling Report", explorative=True)

In [ ]:
profile

In [ ]:
X = df.drop(columns=['survived'])
y = df['survived']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

In [ ]:
num_cols = ['age', 'fare_log', 'sibsp', 'parch']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
log_reg = LogisticRegression(random_state=123)
log_scores = cross_val_score(log_reg, X_train, y_train, cv=5, scoring='accuracy')
log_reg.fit(X_train, y_train)
train_pred_log = log_reg.predict(X_train)
test_pred_log = log_reg.predict(X_test)
print("Cross Validation Accuracy:", log_scores.mean())
print("Train Accuracy:", accuracy_score(y_train, train_pred_log))
print("Test Accuracy:", accuracy_score(y_test, test_pred_log))
print("\nClassification Report:")
print(classification_report(y_test, test_pred_log))

## Logistic Regression showed stable performance with balanced training and testing accuracy, indicating good generalization and minimal overfitting. It served as a strong baseline classification model.

In [ ]:
dt_classifier = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=123
)
dt_scores = cross_val_score(dt_classifier, X_train, y_train, cv=5, scoring='accuracy')
dt_classifier.fit(X_train, y_train)
train_pred_dt = dt_classifier.predict(X_train)
test_pred_dt = dt_classifier.predict(X_test)
print("Cross Validation Accuracy:", dt_scores.mean())
print("Train Accuracy:", accuracy_score(y_train, train_pred_dt))
print("Test Accuracy:", accuracy_score(y_test, test_pred_dt))
print("\nClassification Report:")
print(classification_report(y_test, test_pred_dt))

## After tuning, the Decision Tree achieved the best test accuracy while significantly reducing overfitting. It demonstrated effective learning and good predictive performance.

In [ ]:
rf_classifier = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=3,
    random_state=123
)
rf_scores = cross_val_score(rf_classifier, X_train, y_train, cv=5, scoring='accuracy')
rf_classifier.fit(X_train, y_train)
train_pred_rf = rf_classifier.predict(X_train)
test_pred_rf = rf_classifier.predict(X_test)
print("Cross Validation Accuracy:", rf_scores.mean())
print("Train Accuracy:", accuracy_score(y_train, train_pred_rf))
print("Test Accuracy:", accuracy_score(y_test, test_pred_rf))
print("\nClassification Report:")
print(classification_report(y_test, test_pred_rf))

## Random Forest achieved the highest cross-validation accuracy, demonstrating strong robustness and consistent performance. Although its test accuracy was slightly lower than the tuned Decision Tree, it remained a reliable ensemble model.

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Cross Validation': [
        log_scores.mean(),
        dt_scores.mean(),
        rf_scores.mean()
    ],
    'Train Accuracy': [
        accuracy_score(y_train, train_pred_log),
        accuracy_score(y_train, train_pred_dt),
        accuracy_score(y_train, train_pred_rf)
    ],
    'Test Accuracy': [
        accuracy_score(y_test, test_pred_log),
        accuracy_score(y_test, test_pred_dt),
        accuracy_score(y_test, test_pred_rf)
    ]
})

results.set_index('Model').plot(kind='bar', figsize=(10,6))

plt.title('Model Performance Comparison')
plt.ylabel('Accuracy')
plt.xlabel('Models')
plt.ylim(0.6, 1.0)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.show()